In [5]:
#Setup
import pandas as pd
import numpy as np
from deltalake import DeltaTable, write_deltalake
from datetime import date
import shutil, os

RAW_CSV_PATH = "Sample_-_Superstore.csv"   # put this file in the same folder as the notebook
SCD1_TABLE = "customer_scd1"
SCD2_TABLE = "customer_scd2"
TRACKED_COLS = ["segment", "city", "state", "region", "total_spent"]
SPLIT_QUANTILE = 0.85
TODAY = date.today().isoformat()

for t in (SCD1_TABLE, SCD2_TABLE):
    if os.path.exists(t):
        shutil.rmtree(t)

np.random.seed(42)
print("Environment ready.")

Environment ready.


In [10]:
#Data Loading
raw_orders = pd.read_csv(r"C:\Users\Ritesh\OneDrive\Desktop\Celebal Excellence Intership\Week7_Delta_lake_Assignment\Data\Sample - Superstore.csv", encoding="latin1")
print(f"Raw Superstore rows: {len(raw_orders)}")
print(f"Unique customers: {raw_orders['Customer ID'].nunique()}")
raw_orders.head(10)

Raw Superstore rows: 9994
Unique customers: 793


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164
5,6,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,FUR-FU-10001487,Furniture,Furnishings,Eldon Expressions Wood and Plastic Desk Access...,48.8600,7,0.00,14.1694
6,7,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-AR-10002833,Office Supplies,Art,Newell 322,7.2800,4,0.00,1.9656
7,8,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,TEC-PH-10002275,Technology,Phones,Mitel 5320 IP Phone VoIP phone,907.1520,6,0.20,90.7152
8,9,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-BI-10003910,Office Supplies,Binders,DXL Angle-View Binders with Locking Rings by S...,18.5040,3,0.20,5.7825
9,10,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,OFF-AP-10002892,Office Supplies,Appliances,Belkin F5C206VTEL 6 Outlet Surge,114.9000,5,0.00,34.4700


In [11]:
orders = raw_orders.copy()
orders["order_date"] = pd.to_datetime(orders["Order Date"], format="%m/%d/%Y")
orders = orders.sort_values(["Customer ID", "order_date"]).reset_index(drop=True)
orders["total_spent"] = orders.groupby("Customer ID")["Sales"].cumsum().round(2)

snapshots = orders.rename(columns={
    "Customer ID": "customer_id", "Customer Name": "name", "Segment": "segment",
    "City": "city", "State": "state", "Region": "region", "Postal Code": "postal_code",
})[["customer_id", "name", "segment", "city", "state", "region", "postal_code", "order_date", "total_spent"]]
snapshots = snapshots.rename(columns={"order_date": "snapshot_date"})

cutoff = snapshots["snapshot_date"].quantile(SPLIT_QUANTILE)
print(f"Cutoff date: {cutoff.date()}")

master_snapshots = snapshots[snapshots["snapshot_date"] < cutoff].copy().reset_index(drop=True)
incremental_snapshots = snapshots[snapshots["snapshot_date"] >= cutoff].copy().reset_index(drop=True)

print(f"Master (pre-cutoff) rows: {len(master_snapshots)} ({master_snapshots['customer_id'].nunique()} customers)")
print(f"Incremental (post-cutoff) rows: {len(incremental_snapshots)}")

Cutoff date: 2017-09-11
Master (pre-cutoff) rows: 8489 (789 customers)
Incremental (post-cutoff) rows: 1505


In [12]:
incremental_df = (
    incremental_snapshots.sort_values("snapshot_date")
    .drop_duplicates("customer_id", keep="last")
    .reset_index(drop=True)
)

new_ids_preview = set(incremental_df["customer_id"]) - set(master_snapshots["customer_id"])
print(f"customer_incremental: {len(incremental_df)} rows "
      f"({len(incremental_df) - len(new_ids_preview)} updates, {len(new_ids_preview)} new)")
incremental_df.head(10)

customer_incremental: 494 rows (490 updates, 4 new)


,customer_id,name,segment,city,state,region,postal_code,snapshot_date,total_spent
0,RC-19960,Ryan Crowe,Consumer,Jacksonville,Florida,South,32216,2017-09-11,885.75
1,SS-20410,Shahid Shariari,Consumer,Georgetown,Kentucky,South,40324,2017-09-11,3056.81
2,JM-15865,John Murray,Consumer,Los Angeles,California,West,90036,2017-09-12,7625.08
3,SG-20605,Speros Goranitis,Consumer,Asheville,North Carolina,South,28806,2017-09-13,3124.83
4,TB-21520,Tracy Blumstein,Consumer,Jackson,Michigan,Central,49201,2017-09-14,4737.49
5,DB-13555,Dorothy Badders,Corporate,Raleigh,North Carolina,South,27604,2017-09-14,3834.45
6,PS-18760,Pamela Stobb,Consumer,Philadelphia,Pennsylvania,East,19140,2017-09-14,1812.37
7,CC-12145,Charles Crestani,Consumer,Houston,Texas,Central,77095,2017-09-15,2471.65
8,GM-14500,Gene McClure,Consumer,Seattle,Washington,West,98103,2017-09-15,1255.68
9,CL-11890,Carl Ludwig,Consumer,Everett,Massachusetts,East,2149,2017-09-15,220.31


In [13]:
rng = np.random.default_rng(42)
null_idx = rng.choice(master_snapshots.index, size=int(len(master_snapshots) * 0.015), replace=False)
master_snapshots.loc[null_idx, "segment"] = None
null_idx2 = rng.choice(master_snapshots.index, size=int(len(master_snapshots) * 0.01), replace=False)
master_snapshots.loc[null_idx2, "city"] = None

master_snapshots["snapshot_date"] = master_snapshots["snapshot_date"].dt.strftime("%Y-%m-%d")
incremental_df["snapshot_date"] = incremental_df["snapshot_date"].dt.strftime("%Y-%m-%d")

# IMPORTANT: reset_index before every write_deltalake call, or you'll get a
# "SchemaMismatchError: Cannot cast schema" from a phantom index column
raw_df = master_snapshots.reset_index(drop=True)
incremental_df = incremental_df.reset_index(drop=True)

print(f"customer_master row count: {len(raw_df)}")
raw_df.head(20)

customer_master row count: 8489


,customer_id,name,segment,city,state,region,postal_code,snapshot_date,total_spent
0,AA-10315,Alex Avila,Consumer,San Francisco,California,West,94122,2014-03-31,673.57
1,AA-10315,Alex Avila,Consumer,San Francisco,California,West,94122,2014-03-31,726.55
2,AA-10315,Alex Avila,Consumer,New York City,New York,East,10011,2014-09-15,741.49
3,AA-10315,Alex Avila,Consumer,New York City,New York,East,10011,2014-09-15,756.05
4,AA-10315,Alex Avila,Consumer,San Francisco,California,West,94109,2015-10-04,783.01
5,AA-10315,Alex Avila,Consumer,Round Rock,Texas,Central,78664,2016-03-03,4713.08
6,AA-10315,Alex Avila,Consumer,Round Rock,Texas,Central,78664,2016-03-03,4715.38
7,AA-10315,Alex Avila,Consumer,Round Rock,Texas,Central,78664,2016-03-03,5147.36
8,AA-10315,Alex Avila,Consumer,Round Rock,Texas,Central,78664,2016-03-03,5189.08
9,AA-10315,Alex Avila,Consumer,Minneapolis,Minnesota,Central,55407,2017-06-29,5552.02


In [14]:
write_deltalake(SCD1_TABLE, raw_df, mode="overwrite")

dt = DeltaTable(SCD1_TABLE)
loaded = dt.to_pandas()
print(f"Delta table '{SCD1_TABLE}' created, version {dt.version()}")
print(f"Row count: {len(loaded)}")
print(f"Duplicate customer_id rows: {loaded['customer_id'].duplicated().sum()}")
print(f"Null values: {loaded.isnull().sum().sum()}")

Delta table 'customer_scd1' created, version 0
Row count: 8489
Duplicate customer_id rows: 7700
Null values: 211


In [15]:
#Data Cleaning
current = DeltaTable(SCD1_TABLE).to_pandas()
print("Nulls per column before cleaning:")
print(current.isnull().sum())
print(f"\nDuplicate customer_id rows before cleaning: {current['customer_id'].duplicated().sum()}")

Nulls per column before cleaning:
customer_id        0
name               0
segment          127
city              84
state              0
region             0
postal_code        0
snapshot_date      0
total_spent        0
dtype: int64

Duplicate customer_id rows before cleaning: 7700


In [16]:
#Data cleaning
clean_df = (
    current.sort_values("snapshot_date")
    .drop_duplicates(subset="customer_id", keep="last")
    .reset_index(drop=True)
)
clean_df["segment"] = clean_df["segment"].fillna("Unknown")
clean_df["city"] = clean_df["city"].fillna("Unknown City")
clean_df = clean_df.sort_values("customer_id").reset_index(drop=True)

write_deltalake(SCD1_TABLE, clean_df, mode="overwrite")

cleaned = DeltaTable(SCD1_TABLE).to_pandas().sort_values("customer_id").reset_index(drop=True)
print(f"Row count after cleaning: {len(cleaned)}")
print(f"Remaining duplicate customer_id rows: {cleaned['customer_id'].duplicated().sum()}")
cleaned.head(20)

Row count after cleaning: 789
Remaining duplicate customer_id rows: 0


,customer_id,name,segment,city,state,region,postal_code,snapshot_date,total_spent
0,AA-10315,Alex Avila,Consumer,Minneapolis,Minnesota,Central,55407,2017-06-29,5563.56
1,AA-10375,Allen Armold,Consumer,Providence,Rhode Island,East,2908,2017-09-07,866.56
2,AA-10480,Andrew Allen,Consumer,Concord,North Carolina,South,28027,2017-04-15,1790.51
3,AA-10645,Anna Andreadi,Consumer,Georgetown,Kentucky,South,40324,2016-09-04,5068.70
4,AB-10015,Aaron Bergman,Consumer,Oklahoma City,Oklahoma,Central,73120,2016-11-10,544.20
5,AB-10060,Adam Bellavance,Home Office,Los Angeles,California,West,90004,2017-05-07,4899.35
6,AB-10105,Adrian Barton,Consumer,Bloomington,Illinois,Central,61701,2017-08-03,13037.42
7,AB-10150,Aimee Bixby,Consumer,Long Beach,New York,East,11561,2017-09-04,828.01
8,AB-10165,Alan Barnes,Consumer,Toledo,Ohio,East,43615,2017-04-14,806.36
9,AB-10255,Alejandro Ballentine,Home Office,Los Angeles,California,West,90036,2017-07-17,914.53


In [17]:
scd2_init = cleaned.copy().reset_index(drop=True)
scd2_init["effective_start_date"] = "2014-01-01"
scd2_init["effective_end_date"] = pd.array([None] * len(scd2_init), dtype="string")
scd2_init["is_current"] = True

write_deltalake(SCD2_TABLE, scd2_init, mode="overwrite")
print(f"SCD2 table initialized with {len(DeltaTable(SCD2_TABLE).to_pandas())} rows (all is_current = True)")

SCD2 table initialized with 789 rows (all is_current = True)


In [18]:
#SCD1 Merge
dt1 = DeltaTable(SCD1_TABLE)
v_before = dt1.version()

scd1_result = (
    dt1.merge(
        source=incremental_df,
        predicate="target.customer_id = source.customer_id",
        source_alias="source",
        target_alias="target",
    )
    .when_matched_update_all()
    .when_not_matched_insert_all()
    .execute()
)

print(f"SCD1 table version before merge: {v_before}")
print(f"SCD1 table version after merge:  {DeltaTable(SCD1_TABLE).version()}")
for k, v in scd1_result.items():
    print(f"  {k}: {v}")

SCD1 table version before merge: 1
SCD1 table version after merge:  2
  num_source_rows: 494
  num_target_rows_inserted: 4
  num_target_rows_updated: 490
  num_target_rows_deleted: 0
  num_target_rows_copied: 299
  num_output_rows: 793
  num_target_files_scanned: 1
  num_target_files_skipped_during_scan: 0
  num_target_files_added: 1
  num_target_files_removed: 1
  execution_time_ms: 318
  scan_time_ms: 72
  rewrite_time_ms: 0


In [19]:
sample_updated_id = sorted(set(incremental_df["customer_id"]) & set(cleaned["customer_id"]))[0]
print(f"SCD1 result for customer {sample_updated_id} — old values fully replaced, no trace of history:")
DeltaTable(SCD1_TABLE).to_pandas().query("customer_id == @sample_updated_id")


SCD1 result for customer AA-10375 — old values fully replaced, no trace of history:


,customer_id,name,segment,city,state,region,postal_code,snapshot_date,total_spent
385,AA-10375,Allen Armold,Consumer,New York City,New York,East,10035,2017-12-11,921.47


In [20]:
#SCD2 Merge
dt2 = DeltaTable(SCD2_TABLE)
current_scd2 = dt2.to_pandas()
current_now = current_scd2[current_scd2["is_current"] == True]

compare = incremental_df.merge(
    current_now[["customer_id"] + TRACKED_COLS], on="customer_id", how="left", suffixes=("", "_old")
)
existing_ids = set(current_now["customer_id"])

def is_new_or_changed(row):
    if row["customer_id"] not in existing_ids:
        return True
    return any(row[c] != row[f"{c}_old"] for c in TRACKED_COLS)

compare["is_new_or_changed"] = compare.apply(is_new_or_changed, axis=1)
changed_ids = compare.loc[compare["is_new_or_changed"], "customer_id"].tolist()
print(f"Customers needing a new SCD2 version: {len(changed_ids)} / {len(incremental_df)}")

Customers needing a new SCD2 version: 494 / 494


In [21]:
v_before_scd2 = dt2.version()

expire_result = (
    dt2.merge(
        source=pd.DataFrame({"customer_id": changed_ids}),
        predicate="target.customer_id = source.customer_id AND target.is_current = true",
        source_alias="source",
        target_alias="target",
    )
    .when_matched_update(updates={"is_current": "false", "effective_end_date": f"'{TODAY}'"})
    .execute()
)

new_versions = incremental_df[incremental_df["customer_id"].isin(changed_ids)].copy().reset_index(drop=True)
new_versions["effective_start_date"] = TODAY
new_versions["effective_end_date"] = pd.array([None] * len(new_versions), dtype="string")
new_versions["is_current"] = True
write_deltalake(SCD2_TABLE, new_versions, mode="append")

print(f"SCD2 version before: {v_before_scd2}  →  after: {DeltaTable(SCD2_TABLE).version()}")
print(f"Rows inserted: {len(new_versions)}")

SCD2 version before: 0  →  after: 2
Rows inserted: 494


In [22]:
print(f"Full SCD2 history for customer {sample_updated_id} (old row closed out, new row current):")
cols = ["customer_id", "name", "segment", "city", "state", "region", "total_spent",
        "effective_start_date", "effective_end_date", "is_current"]
DeltaTable(SCD2_TABLE).to_pandas()[cols].query("customer_id == @sample_updated_id").sort_values("effective_start_date")

Full SCD2 history for customer AA-10375 (old row closed out, new row current):


,customer_id,name,segment,city,state,region,total_spent,effective_start_date,effective_end_date,is_current
875,AA-10375,Allen Armold,Consumer,Providence,Rhode Island,East,866.56,2014-01-01,2026-07-26,False
385,AA-10375,Allen Armold,Consumer,New York City,New York,East,921.47,2026-07-26,NaN,True


In [23]:
#Validation
scd1_final = DeltaTable(SCD1_TABLE).to_pandas().sort_values("customer_id").reset_index(drop=True)
new_ids_preview = sorted(set(incremental_df["customer_id"]) - set(cleaned["customer_id"]))
expected_scd1_rows = len(cleaned) + len(new_ids_preview)
dup_count_scd1 = scd1_final["customer_id"].duplicated().sum()

print("SCD1 VALIDATION")
print(f"Row count check: {'PASS' if len(scd1_final) == expected_scd1_rows else 'FAIL'}")
print(f"Duplicate check: {'PASS' if dup_count_scd1 == 0 else 'FAIL'}")

SCD1 VALIDATION
Row count check: PASS
Duplicate check: PASS


In [24]:
scd2_final = DeltaTable(SCD2_TABLE).to_pandas().sort_values(["customer_id", "effective_start_date"]).reset_index(drop=True)
expected_scd2_rows = len(cleaned) + len(changed_ids)
current_rows = scd2_final[scd2_final["is_current"] == True]
dup_current = current_rows["customer_id"].duplicated().sum()

print("SCD2 VALIDATION")
print(f"Row count check: {'PASS' if len(scd2_final) == expected_scd2_rows else 'FAIL'}")
print(f"Exactly-one-current-row check: {'PASS' if dup_current == 0 else 'FAIL'}")

SCD2 VALIDATION
Row count check: PASS
Exactly-one-current-row check: PASS


In [ ]:
#